# SAPCAR manifest API

This notebook describes the public library API for parsing, validating, and crafting SAPCAR manifests.

## Opening an archive and locating its manifest

`SAPCARArchive.find_manifests()` recognizes `SIGNATURE.SMF`, `MANIFEST.MF`, and `MANIFEST`. `read_manifest()` decompresses and parses one recognized member into a `SAPCARManifest`. For standalone bytes, use `SAPCARManifest.from_bytes()`.

In [1]:
from pathlib import Path

from pysap.SAPCAR import SAPCARArchive

fixture_dir = Path("../../tests/data").resolve()
archive_path = fixture_dir / "manifest_signed.sar"

In [2]:
with archive_path.open("rb") as archive_file:
    archive = SAPCARArchive(archive_file, mode="r")
    manifest_names = [
        name.decode("utf-8") if isinstance(name, bytes) else name
        for name in archive.find_manifests()
    ]
    manifest = archive.read_manifest()

manifest_names, [(entry.filename, entry.digest_algorithm) for entry in manifest.entries]

(['SIGNATURE.SMF'], [('payload.txt', 'sha256')])

`SAPCARManifest` retains parsed header metadata, normalized entries, the exact bytes covered by the detached signature in `signed_data`, and the decoded CMS object in `signature_data`. `as_dict()` provides a serializable representation; the raw fields remain available when required.

## Content and signature validation

`SAPCARArchive.validate_manifest()` separates archive integrity from signature trust. It checks manifest syntax, safe normalized names, duplicate members, missing or unlisted members, decompression checksums, declared sizes, and content digests. Use `SAPCARManifest.verify_signature()` when only the detached signature needs verification.

In [3]:
with archive_path.open("rb") as archive_file:
    archive = SAPCARArchive(archive_file, mode="r")
    report = archive.validate_manifest(
        strict=True,
        certificate_dir=str(fixture_dir),
    )

{key: report[key] for key in (
    "manifest_status", "integrity_valid", "signature_status", "valid"
)}

{'manifest_status': 'present',
 'integrity_valid': True,
 'signature_status': 'valid',
 'valid': True}

The report keeps distinct results:

- `integrity_valid` covers manifest-to-archive consistency.
- `signature_status` is `valid`, `invalid`, `unverified`, or `not-present`.
- `valid` follows integrity in normal mode. In strict mode it additionally requires a trusted, valid signature and strong per-member digests.

Detailed lists such as `missing`, `unexpected`, `size_mismatches`, `digest_mismatches`, and `malformed_entries` identify the reason for a failure.

## How detached signature validation works

For a `PKCS7-TSTAMP` manifest, validation proceeds in four stages:

1. The parser preserves every byte before `-----BEGIN SIGNATURE-----`; those exact bytes are the detached signed content.
2. The CMS signed attributes are decoded. Their message digest is compared with the manifest's `signed_data`, then the signer signature is verified with the certificate embedded in the CMS object.
3. The RFC 3161-shaped timestamp token is checked against the signer signature. Its content digest and timestamp signature must both match.
4. The signer and timestamp certificates are matched against trust material. An explicit trust store or certificate directory overrides the SAP certificates embedded by the library; without an override, the embedded SAPCAR certificates mimic native-tool trust behavior.

A cryptographically correct signature without matching trust material is `unverified`, not `invalid`. Trust matching supports a directly pinned certificate or direct issuer. It does not perform complete PKI path construction, policy processing, validity-at-timestamp evaluation, or revocation checking.

## Crafting a signed manifest

`SAPCARArchive.sign_manifest()` signs all regular archive members and replaces an existing recognized manifest. The certificate and unencrypted RSA private key can be paths or loaded cryptography objects. A separate timestamp certificate and key are optional. The following example is intentionally not executed because it writes a new archive:

```python
archive = SAPCARArchive("output.sar", mode="w")
archive.add_file("payload.bin", archive_filename="payload.bin")
archive.sign_manifest(
    certificate="signer.crt",
    private_key="signer.key",
    timestamp_certificate="tsa.crt",
    timestamp_private_key="tsa.key",
)
archive.write()
archive.close()
```

The generated timestamp is local and proves possession of the configured key; it is not evidence from an independent timestamp service.